In [ ]:
import sys
import os
import boto3

from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

# Spark / Glue contexts
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print(spark.version)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

3.3.0-amzn-1

In [11]:
%%bash
aws sts get-caller-identity --profile glue_dev

{
    "Account": "049618906779", 
    "UserId": "AIDAQXDMEN2NROHG5GKH4", 
    "Arn": "arn:aws:iam::049618906779:user/tariff_project"
}


In [4]:
import boto3

glue = boto3.client("glue")

glue.get_databases()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'DatabaseList': [], 'ResponseMetadata': {'RequestId': 'a5cdf899-1677-43eb-afec-a871cc5655ed', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 07 Dec 2025 06:43:09 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '19', 'connection': 'keep-alive', 'x-amzn-requestid': 'a5cdf899-1677-43eb-afec-a871cc5655ed', 'cache-control': 'no-cache'}, 'RetryAttempts': 0}}

In [49]:
from pyspark.sql import SparkSession

REGION = "us-east-1"
WAREHOUSE = "s3://dummy-lakehouse/"   # must be a REAL S3 bucket created in AWS

spark = (
    SparkSession.builder
        # Iceberg extension
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

        # Glue catalog configuration
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.warehouse", WAREHOUSE)
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.lock-impl", "org.apache.iceberg.aws.glue.DynamoLockManager")
        .config("spark.sql.catalog.glue_catalog.region", REGION)

        # AWS SDK settings (optional but recommended)
        .config("spark.sql.catalog.glue_catalog.s3.signing-region", REGION)

        # Spark Hadoop AWS configuration
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
                "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

        .config("spark.hadoop.fs.s3a.region", REGION)
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
        .config("spark.hadoop.fs.s3a.path.style.access", "false")  # IMPORTANT for real S3

        .getOrCreate()
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [48]:
print(spark.sparkContext._jsc.hadoopConfiguration().get("fs.s3a.endpoint"))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3.amazonaws.com

In [27]:
spark.sql("""
    CREATE DATABASE IF NOT EXISTS glue_catalog.lakehouse_db
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [50]:
spark.sql("""
    CREATE TABLE glue_catalog.lakehouse_db.test_table (
        id INT,
        name STRING
    )
    USING iceberg
    LOCATION 's3a://dummy-lakehouse/lakehouse_db/test_table'
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [52]:
spark.sql("""
    INSERT INTO glue_catalog.lakehouse_db.test_table 
    VALUES (1, 'alice'), (2, 'bob'), (3, 'charlie')
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [54]:
spark.sql("SELECT * FROM glue_catalog.lakehouse_db.test_table").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+-------+
| id|   name|
+---+-------+
|  1|  alice|
|  2|    bob|
|  3|charlie|
+---+-------+